# Segmented Whittle Likelihood Pipeline

Instead of multiplying the data by a gap window (which creates a singular
Fourier-domain covariance), this notebook **splits the processed data into
contiguous clean segments** delimited by the extended mask, applies a mild
Tukey taper to each chunk independently, and computes a standard Whittle
periodogram per segment.

Each segment's periodogram is then compared against the Mojito L1 noise PSD
estimate. Because no zeros are injected, the standard diagonal Whittle
likelihood $\mathcal{L} \propto \prod_k S_k^{-1} \exp(-|d_k|^2 / S_k)$ is
well-defined for every segment.

| Step | Operation | Notes |
|------|-----------|-------|
| 1 | Generate gap mask | `lisaglitch` + `lisagap` |
| 2 | Apply mask to raw data | `gaps.apply_raw_mask` |
| 3 | Run processing pipeline | filter → downsample → trim → segment |
| 4 | Compute extended mask | `gaps.compute_extended_mask` (filter leakage) |
| 5 | Extract clean segments | Split at `extended_mask == 0` boundaries |
| 6 | Taper & FFT per segment | Mild Tukey taper, standard periodogram |
| 7 | Compare vs L1 estimate | Overlay on periodogram plot |

In [ ]:
import logging

import numpy as np
import matplotlib.pyplot as plt

from lisaglitch import GapMaskGenerator
from lisagap import GapWindowGenerator
from scipy.signal import windows

import MojitoProcessor as mp
import MojitoProcessor.gaps as gaps

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

## 1. Load Data

In [ ]:
mojito_data_file = (
    "../Mojito_Data/NOISE_731d_0.25s_L1_source0_0_20251206T220508924302Z.h5"
)

# Set to a float (days) to load only a subset of the file
load_days = None

data = mp.io.load_file(mojito_data_file, load_days=load_days)

n_samples = len(data["tdis"]["X"])
duration = n_samples * data["dt"]
print(f"Loaded: {n_samples:,} samples @ {data['fs']} Hz ({duration / 86400:.2f} days)")
print(f"TDI channels: {list(data['tdis'].keys())}")

## 2. Generate Gap Mask

Use `lisaglitch.GapMaskGenerator` to define planned gaps (antenna repointing),
then wrap with `lisagap.GapWindowGenerator` to produce a smooth tapering mask.
A Tukey window is applied over the whole dataset to bring the endpoints to zero.

In [ ]:
# ── Gap schedule ──────────────────────────────────────────────────────────────
gap_definitions = {
    "planned": {
        "antenna_repointing": {
            "rate_per_year": 26,
            "duration_hr": 7,
        }
    },
    "unplanned": {},
}

taper_definitions = {
    "planned": {
        "antenna_repointing": {"lobe_lengths_hr": 3.0},
    },
    "unplanned": {},
}

# ── Build smoothed mask at the raw sampling rate ──────────────────────────────
gap_gen = GapMaskGenerator(
    sim_t=data["t_tdi"],
    gap_definitions=gap_definitions,
    treat_as_nan=False,
)
window_func = GapWindowGenerator(gap_gen)
smoothed_mask = window_func.generate_window(
    include_planned=True,
    include_unplanned=False,
    apply_tapering=True,
    taper_definitions=taper_definitions,
)[0]

# Multiply by a dataset-wide Tukey window to bring the endpoints to zero
smoothed_mask = windows.tukey(len(smoothed_mask), alpha=0.01) * smoothed_mask

print(f"Gap fraction in raw mask: {1 - smoothed_mask.mean():.4%}")

## 3. Apply Mask to Raw Data

`gaps.apply_raw_mask` deep-copies the data dict and zeros out every TDI channel
where the mask is zero. The original `data` dict is unchanged.

In [ ]:
data_gapped = gaps.apply_raw_mask(data, smoothed_mask)

plt.figure(figsize=(14, 3))
plt.plot(data["t_tdi"][::50] / 3600, data_gapped["tdis"]["X"][::50], label="X (gapped)")
plt.plot(
    data["t_tdi"][::50] / 3600,
    np.max(np.abs(data["tdis"]["X"])) * smoothed_mask[::50],
    label="Smoothed mask",
    alpha=0.7,
)
plt.xlabel("Time (hours)")
plt.ylabel("Signal")
plt.title("TDI-X with gap mask applied")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Run the Processing Pipeline

All pipeline parameters are configurable here. The pipeline runs in this order:

```
bandpass/highpass filter → downsample → trim → truncate → window
```

The time-domain data is normalised by the laser central frequency inside
`process_pipeline` so the output is dimensionless fractional frequency.

In [ ]:
# ── Pipeline parameters ───────────────────────────────────────────────────────
downsample_kwargs = {
    "target_fs": 0.2,  # Hz — target sampling rate
    "kaiser_window": 31.0,  # Kaiser β for anti-aliasing
}
filter_kwargs = {
    "highpass_cutoff": 5e-5,  # Hz
    "lowpass_cutoff": 0.8 * downsample_kwargs["target_fs"],  # Hz
    "order": 2,
}
trim_kwargs = {
    "fraction": 0.015,  # fraction trimmed from each end after downsampling
}
truncate_kwargs = {
    "days": 90.0,  # split dataset into 15-day segments
}
window_kwargs = {
    "window": "tukey",
    "alpha": 0.0,
}
# ─────────────────────────────────────────────────────────────────────────────

processed_segments = mp.process_pipeline(
    data_gapped,
    downsample_kwargs=downsample_kwargs,
    filter_kwargs=filter_kwargs,
    trim_kwargs=trim_kwargs,
    truncate_kwargs=truncate_kwargs,
    window_kwargs=window_kwargs,
)

sp_0 = processed_segments["segment0"]
print(f"Processed segments: {list(processed_segments.keys())}")
print(f"segment0: N={sp_0.N}, fs={sp_0.fs} Hz, T={sp_0.T / 86400:.2f} days")

## 5. Compute Extended Mask

`gaps.compute_extended_mask` mirrors the pipeline's downsampling and trimming,
filters `1 − mask` through the same Butterworth, and flags samples where the
absolute leakage exceeds `contamination_threshold` as excluded.

Both outputs are kept for inspection before applying any further steps.

> **Note:** `fs_raw=data["fs"]` must be the raw (pre-downsample) sampling rate.
> `smoothed_mask` spans the full dataset so this cannot be inferred from the segment alone.

In [ ]:
extended_mask, gap_contamination = gaps.compute_extended_mask(
    smoothed_mask,
    sp_0,
    filter_kwargs,
    downsample_kwargs,
    trim_kwargs,
    fs_raw=data["fs"],
    contamination_threshold=1e-4,
    min_clean_hours=12.0,
)

ext_gap = 1 - extended_mask.mean()
print(f"Extended gap fraction : {ext_gap:.4%}")
print(
    f"Mask length: {len(extended_mask):,}  sp_0.N: {sp_0.N:,}  match: {len(extended_mask) == sp_0.N}"
)

In [ ]:
# ── Presentation build-up plots: taper → corruption → extended mask ───────────
# Align smoothed_mask with sp_0 by replicating the pipeline's downsample+trim.
# compute_extended_mask trims fraction/2 from each end (not fraction from one end).
_ds_factor = round(data["fs"] / downsample_kwargs["target_fs"])  # 20
_n_ds_total = len(smoothed_mask) // _ds_factor  # ~12,623,260
_trim_n = int(round(_n_ds_total * trim_kwargs["fraction"] / 2))  # 94,674
_mask_ds = smoothed_mask[::_ds_factor][_trim_n : _trim_n + sp_0.N]

# Focus on a region containing a representative gap
_gap_locs = np.where(extended_mask == 0)[0]
_focus_center = _gap_locs[len(_gap_locs) // 4]  # ~25 % through segment 0
_window_samp = int(50 * 3600 * sp_0.fs)  # ±50 h either side
_i0 = max(0, _focus_center - _window_samp)
_i1 = min(sp_0.N, _focus_center + _window_samp)
_t_hr = (sp_0.t[_i0:_i1] - sp_0.t[_i0]) / 3600  # hours from window start

_ylim = (-0.05, 1.15)
_figsize = (14, 4)
_fs_label = 18  # axis labels
_fs_title = 20  # titles
_fs_tick = 16  # tick labels
_fs_legend = 16  # legend

# ── (a) Original tapering function ────────────────────────────────────────────
fig_a, ax_a = plt.subplots(figsize=_figsize)
ax_a.plot(_t_hr, _mask_ds[_i0:_i1], color="C0", linewidth=1.5)
ax_a.set_ylim(*_ylim)
ax_a.set_xlabel("Time [hours]", fontsize=_fs_label)
ax_a.set_ylabel("Window value", fontsize=_fs_label)
ax_a.set_title("(a) Original tapering function", fontsize=_fs_title, fontweight="bold")
ax_a.grid(True, alpha=0.3)
ax_a.tick_params(axis="both", which="major", labelsize=_fs_tick)
plt.tight_layout()
plt.show()

# ── (b) Original tapering function + filter corruption ────────────────────────
fig_b, ax_b = plt.subplots(figsize=_figsize)
ax_b.plot(
    _t_hr,
    _mask_ds[_i0:_i1],
    color="C0",
    linewidth=1.5,
    label="Original taper",
    alpha=0.5,
)
ax_b.plot(
    _t_hr, gap_contamination[_i0:_i1], color="C3", linewidth=1.5, label="Filter leakage"
)
ax_b.set_ylim(*_ylim)
ax_b.set_xlabel("Time [hours]", fontsize=_fs_label)
ax_b.set_ylabel("Window value", fontsize=_fs_label)
ax_b.set_title(
    "(b) Original tapering function + filter corruption",
    fontsize=_fs_title,
    fontweight="bold",
)
ax_b.legend(fontsize=_fs_legend, loc="upper right")
ax_b.grid(True, alpha=0.3)
ax_b.tick_params(axis="both", which="major", labelsize=_fs_tick)
plt.tight_layout()
plt.show()

# ── (c) Original tapering + corruption + extended window ──────────────────────
fig_c, ax_c = plt.subplots(figsize=_figsize)
ax_c.plot(
    _t_hr,
    _mask_ds[_i0:_i1],
    color="C0",
    linewidth=1.5,
    label="Original taper",
    alpha=0.4,
)
ax_c.plot(
    _t_hr,
    gap_contamination[_i0:_i1],
    color="C3",
    linewidth=1.5,
    label="Filter leakage",
    alpha=0.6,
)
ax_c.plot(
    _t_hr, extended_mask[_i0:_i1], color="C2", linewidth=2.0, label="Extended mask"
)
ax_c.set_ylim(*_ylim)
ax_c.set_xlabel("Time [hours]", fontsize=_fs_label)
ax_c.set_ylabel("Window value", fontsize=_fs_label)
ax_c.set_title(
    "(c) Original tapering + corruption + extended window function",
    fontsize=_fs_title,
    fontweight="bold",
)
ax_c.legend(fontsize=_fs_legend, loc="upper right")
ax_c.grid(True, alpha=0.3)
ax_c.tick_params(axis="both", which="major", labelsize=_fs_tick)
plt.tight_layout()
plt.show()

## 6. Extract Clean Segments

Split the processed time series at gap boundaries defined by `extended_mask`.
Each contiguous run of `1`s becomes an independent clean segment.  Short
segments (below `min_segment_hours`) are discarded.

In [ ]:
from scipy.signal import windows as sp_windows

min_segment_hours = 12.0  # discard clean stretches shorter than this
tukey_alpha = 0.05  # mild taper for each clean chunk

# ── Extract clean segments using gaps module ──────────────────────────────────
sp_0_segments = gaps.extract_clean_segments(
    sp_0,
    extended_mask,
    min_clean_hours=min_segment_hours,
)

# Index pairs for time-domain plotting
diff = np.diff(np.concatenate([[0], extended_mask.astype(int), [0]]))
starts = np.where(diff == 1)[0]
ends = np.where(diff == -1)[0]
min_samples = int(min_segment_hours * 3600 * sp_0.fs)
clean_segments = [(s, e) for s, e in zip(starts, ends) if (e - s) >= min_samples]

print(f"Total contiguous clean runs: {len(starts)}")
print(f"Kept (≥ {min_segment_hours} h): {len(sp_0_segments)}")
print()
for name, seg in sp_0_segments.items():
    dur_hr = seg.T / 3600
    t0_days = seg.t0 / 86400 if seg.t0 is not None else 0.0
    print(
        f"  {name}: N={seg.N:>6,}, fs={seg.fs} Hz, "
        f"t0={t0_days:.4f} d, duration={dur_hr:.1f} h ({dur_hr / 24:.2f} d)"
    )

In [ ]:
# ── Time-domain view of clean chunks (XYZ): raw + tapered ─────────────────────
from scipy.signal import windows as sp_windows

t_days = (sp_0.t - sp_0.t[0]) / 86400

fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True)

for row, ch in enumerate(["X", "Y", "Z"]):
    ax = axes[row]
    # Full processed trace in light grey
    ax.plot(t_days, sp_0.data[ch], color="0.85", linewidth=0.3, zorder=0)

    # Overlay each clean chunk: raw (faint) + tapered (solid)
    for i, (s, e) in enumerate(clean_segments):
        n_chunk = e - s
        w_chunk = sp_windows.tukey(n_chunk, alpha=tukey_alpha)
        raw = sp_0.data[ch][s:e]
        tapered = raw * w_chunk

        color = f"C{i % 10}"
        ax.plot(t_days[s:e], raw, linewidth=0.3, color=color, alpha=0.35)
        ax.plot(
            t_days[s:e],
            tapered,
            linewidth=0.5,
            color=color,
            label=f"chunk {i}" if row == 0 else None,
        )

    # Shade gap regions
    gap_diff = np.diff(np.concatenate([[1], extended_mask.astype(int), [1]]))
    gap_starts = np.where(gap_diff == -1)[0]
    gap_ends = np.where(gap_diff == 1)[0]
    for gs, ge in zip(gap_starts, gap_ends):
        ax.axvspan(
            t_days[min(gs, len(t_days) - 1)],
            t_days[min(ge - 1, len(t_days) - 1)],
            alpha=0.15,
            color="red",
            zorder=-1,
        )

    ax.set_ylabel(f"TDI-{ch}", fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="both", which="major", labelsize=12)

axes[0].legend(loc="upper right", fontsize=10, ncol=min(len(clean_segments), 5))
axes[2].set_xlabel("Time [days]", fontsize=14)
fig.suptitle(
    f"Clean segments: raw (faint) vs Tukey-tapered α={tukey_alpha} (solid)",
    fontsize=15,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Apply Tukey taper to each clean segment ───────────────────────────────────
for name, seg in sp_0_segments.items():
    seg.apply_window("tukey", alpha=tukey_alpha)
    print(f"{name}: windowed (Tukey α={tukey_alpha}), N={seg.N:,}")

## 7. Per-Segment Periodograms

Each windowed segment is converted to AET via `sp.to_aet()` and the
one-sided periodogram is computed using `sp.periodogram()`.

In [ ]:
CENTRAL_FREQ = data["metadata"]["laser_frequency"]

# ── L1 noise model ────────────────────────────────────────────────────────────
noise_cov_xyz = data["noise_estimates"]["xyz"]
noise_cov_aet = data["noise_estimates"]["aet"]
noise_freqs = np.logspace(-5, 0, noise_cov_xyz[0].shape[0])

l1_xyz = {
    ch: noise_cov_xyz[0][:, i, i] / CENTRAL_FREQ**2
    for i, ch in enumerate(["X", "Y", "Z"])
}
l1_aet = {
    ch: noise_cov_aet[0][:, i, i] / CENTRAL_FREQ**2
    for i, ch in enumerate(["A", "E", "T"])
}

# ── Compute periodograms using SignalProcessor methods ────────────────────────
chunk_results = []
for name, seg in sp_0_segments.items():
    freqs, psd_xyz = seg.periodogram()
    seg_aet = seg.to_aet()
    _, psd_aet = seg_aet.periodogram()

    chunk_results.append(
        {
            "name": name,
            "N": seg.N,
            "freqs": freqs,
            "psd_xyz": psd_xyz,
            "psd_aet": psd_aet,
        }
    )
    print(
        f"{name}: N={seg.N:,}, df={freqs[1]:.2e} Hz, " f"duration={seg.T / 86400:.2f} d"
    )

## 8. Periodogram vs L1 Estimate — First Two Clean Chunks

Each panel overlays the per-chunk periodogram against the Mojito L1 noise
model.  Because these are gap-free stretches with only a mild Tukey taper,
the standard Whittle likelihood applies directly.

In [ ]:
nyquist = sp_0.fs / 2
xyz_colors = ["C0", "C1", "C2"]
aet_colors = ["#e377c2", "#9467bd", "#8c564b"]

n_chunks = min(4, len(chunk_results))
fig, axes = plt.subplots(
    3, n_chunks, figsize=(8 * n_chunks, 10), sharex=False, sharey=True, squeeze=False
)

for col, cr in enumerate(chunk_results[:n_chunks]):
    f = cr["freqs"]
    dur_d = cr["N"] * sp_0.dt / 86400
    for row, ch in enumerate(["X", "Y", "Z"]):
        ax = axes[row, col]
        ax.loglog(
            f[1:],
            cr["psd_xyz"][ch][1:],
            linewidth=0.8,
            color=xyz_colors[row],
            label=f"TDI-{ch}",
        )
        ax.loglog(
            noise_freqs,
            l1_xyz[ch],
            ls="--",
            color="red",
            linewidth=2.0,
            label="Mojito L1",
        )
        ax.axvspan(1e-4, 1e-1, alpha=0.12, color="green", label="Science band")
        ax.set_xlim(1e-5, nyquist)
        ax.set_ylim(1e-53, 1e-35)
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(loc="upper left", fontsize=11, framealpha=0.95)
        ax.tick_params(axis="both", which="major", labelsize=14)
        if col == 0:
            ax.set_ylabel("PSD [1/Hz]", fontsize=16)
    axes[0, col].set_title(
        f"Chunk {col}  ({dur_d:.2f} d, N={cr['N']:,})", fontsize=14, fontweight="bold"
    )
    axes[2, col].set_xlabel("Frequency [Hz]", fontsize=16)

fig.suptitle(
    "XYZ — Clean-Segment Periodogram vs L1 Estimate\n"
    f"(Tukey α={tukey_alpha}, min segment={min_segment_hours} h)",
    fontsize=16,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    3, n_chunks, figsize=(8 * n_chunks, 10), sharex=False, sharey=True, squeeze=False
)

for col, cr in enumerate(chunk_results[:n_chunks]):
    f = cr["freqs"]
    dur_d = cr["N"] * sp_0.dt / 86400
    for row, ch in enumerate(["A", "E", "T"]):
        ax = axes[row, col]
        ax.loglog(
            f[1:],
            cr["psd_aet"][ch][1:],
            linewidth=0.8,
            color=aet_colors[row],
            label=f"TDI-{ch}",
        )
        ax.loglog(
            noise_freqs,
            l1_aet[ch],
            ls="--",
            color="red",
            linewidth=2.0,
            label="Mojito L1",
        )
        ax.axvspan(1e-4, 1e-1, alpha=0.12, color="green", label="Science band")
        ax.set_xlim(1e-5, nyquist)
        ax.set_ylim(1e-53, 1e-35)
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(loc="upper left", fontsize=11, framealpha=0.95)
        ax.tick_params(axis="both", which="major", labelsize=14)
        if col == 0:
            ax.set_ylabel("PSD [1/Hz]", fontsize=16)
    axes[0, col].set_title(
        f"Chunk {col}  ({dur_d:.2f} d, N={cr['N']:,})", fontsize=14, fontweight="bold"
    )
    axes[2, col].set_xlabel("Frequency [Hz]", fontsize=16)

fig.suptitle(
    "AET — Clean-Segment Periodogram vs L1 Estimate\n"
    f"(Tukey α={tukey_alpha}, min segment={min_segment_hours} h)",
    fontsize=16,
    fontweight="bold",
)
plt.tight_layout()
plt.show()